In [3]:
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

In [4]:
X_train = pd.read_csv("../data/processed/X_train_smote.csv")
X_test = pd.read_csv("../data/processed/X_test.csv")

y_train = pd.read_csv("../data/processed/y_train_smote.csv").squeeze()
y_test = pd.read_csv("../data/processed/y_test.csv").squeeze()

In [5]:
mlflow.set_experiment("Credit Card Fraud Detection")

2026/07/28 16:08:00 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/07/28 16:08:00 INFO mlflow.store.db.utils: Updating database tables
2026/07/28 16:08:02 INFO mlflow.tracking.fluent: Experiment with name 'Credit Card Fraud Detection' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///g:/jusu/cv/DS/Credit-Card-Fraud-Detection/notebooks/mlruns/1', creation_time=1785235082681, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1785235082681, lifecycle_stage='active', name='Credit Card Fraud Detection', tags={}, trace_location=None, workspace='default'>

In [6]:
with mlflow.start_run(run_name="XGBoost_Baseline"):

    model = XGBClassifier(
        random_state=42,
        n_estimators=300,
        max_depth=5,
        learning_rate=0.1,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

In [7]:
mlflow.log_param("Model", "XGBoost")
mlflow.log_param("n_estimators", 300)
mlflow.log_param("max_depth", 5)
mlflow.log_param("learning_rate", 0.1)
mlflow.log_param("random_state", 42)

42

In [8]:
accuracy = accuracy_score(y_test, predictions)

precision = precision_score(y_test, predictions)

recall = recall_score(y_test, predictions)

f1 = f1_score(y_test, predictions)

roc_auc = roc_auc_score(y_test, probabilities)

In [9]:
mlflow.log_metric("Accuracy", accuracy)

mlflow.log_metric("Precision", precision)

mlflow.log_metric("Recall", recall)

mlflow.log_metric("F1 Score", f1)

mlflow.log_metric("ROC AUC", roc_auc)

In [13]:
mlflow.sklearn.log_model(
    sk_model=model,
    artifact_path="model",
    skops_trusted_types=[
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBClassifier",
    ],
)

2026/07/28 16:12:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


In [14]:
results = pd.DataFrame({

    "Metric":[
        "Accuracy",
        "Precision",
        "Recall",
        "F1",
        "ROC AUC"
    ],

    "Value":[
        accuracy,
        precision,
        recall,
        f1,
        roc_auc
    ]
})

results.to_csv(
    "../reports/mlflow_summary.csv",
    index=False
)

mlflow.log_artifact("../reports/mlflow_summary.csv")

In [15]:
mlflow.log_artifact("../images/confusion_matrix.png")
mlflow.log_artifact("../images/roc_curve.png")
mlflow.log_artifact("../images/precision_recall_curve.png")

In [17]:
mlflow.sklearn.log_model(
    sk_model=model,
    artifact_path="model",
    registered_model_name="CreditCardFraudDetector",
    skops_trusted_types=[
        "xgboost.core.Booster",
        "xgboost.sklearn.XGBClassifier",
    ],
)

2026/07/28 16:14:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Successfully registered model 'CreditCardFraudDetector'.
Created version '1' of model 'CreditCardFraudDetector'.


In [19]:
mlflow.end_run()

with mlflow.start_run(run_name="XGBoost_Depth7"):
    model = XGBClassifier(
        random_state=42,
        max_depth=7,
        n_estimators=400,
        learning_rate=0.05,
        eval_metric="logloss",
    )

    # train, evaluate, log parameters/metrics/model

In [23]:
joblib.dump(model, "../models/final_model.pkl")

['../models/final_model.pkl']

In [24]:
import joblib

loaded_model = joblib.load("../models/final_model.pkl")